# Coefficient / Open Philanthropy Grant CSV — Data Audit

This notebook audits `data/raw/op_grants_full.csv` step by step, so we can agree on the data model **before** running the full pipeline.

**Key questions we work through here:**
1. What does the raw CSV actually look like?
2. Which rows are in scope?
3. What are the edge cases: blank org names, person names, mismatches between org and grant title?
4. Should a node represent a **grant** (one row) or an **organisation** (multiple rows grouped)?
5. How should regex be used — entity resolution only, not filtering?
6. What does the clean, agreed node list look like before we touch any pipeline code?

In [ ]:
import csv, re, json
import pandas as pd
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 200)

CSV_PATH = '../data/raw/op_grants_full.csv'
df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
print(f'Total rows in CSV: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 1. Scope: which rows are relevant?

The CSV has 36 focus areas. We care about the biosafety+AI intersection only.

**In-scope focus areas (used for the current graph):**
- `Biosecurity & Pandemic Preparedness` → all rows included, no org filter
- `Science Supporting Biosecurity and Pandemic Preparedness` → all rows included
- `Navigating Transformative AI` → only curated org list (too many unrelated grants otherwise)
- `Global Catastrophic Risks` / `GCR Capacity Building` → only curated org list + bio/AI keyword

**Out-of-scope (examples):** Farm Animal Welfare, Housing Policy Reform, Criminal Justice Reform, etc.

In [ ]:
BIO_FOCUS = {'Biosecurity & Pandemic Preparedness', 'Science Supporting Biosecurity and Pandemic Preparedness'}
AI_FOCUS  = {'Navigating Transformative AI'}
GCR_FOCUS = {'Global Catastrophic Risks', 'Global Catastrophic Risks Capacity Building'}
ALL_FOCUS = BIO_FOCUS | AI_FOCUS | GCR_FOCUS

in_scope = df[df['Focus Area'].isin(ALL_FOCUS)].copy()
bio_scope = df[df['Focus Area'].isin(BIO_FOCUS)].copy()

print(f'All in-scope rows (any relevant FA):  {len(in_scope)}')
print(f'Biosecurity-only rows (no org filter): {len(bio_scope)}')
print(f'\nBreakdown by focus area:')
in_scope['Focus Area'].value_counts()

## 2. The `Organization Name` column — how reliable is it?

The CSV has two columns that both carry entity information:
- `Organization Name` — usually the grantee org, sometimes blank, sometimes abbreviated
- `Grant` — always populated, usually `"<Org> — <Purpose>"` format

Let's look at the three problem classes.

In [ ]:
# Problem class 1: blank org name
blank_org = bio_scope[bio_scope['Organization Name'].isna() | (bio_scope['Organization Name'].str.strip() == '')]
print(f'Biosecurity rows with BLANK org name: {len(blank_org)}')
blank_org[['Grant', 'Amount', 'Date']].head(10)

In [ ]:
# Problem class 2: org name is a person's name (no institutional affiliation shown)
# Pattern: two capitalised words, possibly with a period
person_re = re.compile(r'^[A-Z][a-zA-Z\-]+\.? [A-Z][a-zA-Z\-]+\.?$')
is_person = bio_scope['Organization Name'].fillna('').str.match(person_re)
person_rows = bio_scope[is_person]
print(f'Biosecurity rows where org name looks like a PERSON: {len(person_rows)}')
person_rows[['Organization Name', 'Grant', 'Amount', 'Date']]

In [ ]:
# Problem class 3: org name is just a truncated or abbreviated form of the grant title
# e.g. org = "RAND", grant = "RAND Corporation — Biosecurity Policy"
# We detect these by checking if the org name appears at the start of the grant name
mismatch_rows = []
for _, r in bio_scope.dropna(subset=['Organization Name']).iterrows():
    org = r['Organization Name'].strip()
    grant = r['Grant'].strip()
    # If grant starts with org, they basically match — flag the ones that DON'T
    if org and not grant.lower().startswith(org.lower()[:10]):
        mismatch_rows.append({'Organization Name': org, 'Grant': grant, 'Amount': r['Amount'], 'Date': r['Date']})

mismatch_df = pd.DataFrame(mismatch_rows)
print(f'Biosecurity rows where org name and grant title seem to diverge: {len(mismatch_df)}')
mismatch_df.head(20)

## 3. Node granularity question: grant-level vs org-level?

**Option A — Org-level nodes (current approach):**
- One node per unique organization
- Multiple grants from same org → multiple edges to same node
- Node label: `"SecureBio"`, `"Johns Hopkins Center for Health Security"`
- Pros: cleaner graph, easier to see total funding per org
- Cons: lose grant-level detail; "Funding for Projects to Estimate Biological Risk" is not an org

**Option B — Grant-level nodes:**
- One node per CSV row, label = full grant title
- e.g. `"Jake Pencharz — Metagenomic Sequencing Research"`
- Pros: preserves all context; no information loss
- Cons: 191+ grant nodes just from OP → very dense funding layer

**Option C (recommended) — Org-level nodes, with grant titles stored as edge metadata:**
- Node = the organisation (label: org name)
- Edge = the specific grant (edge has `grant_title`, `amount`, `date` as attributes)
- Full grant title is visible when you click an edge or the org node
- Hover on an org node → see list of all grant titles it received

Below we check how many distinct entities that gives us, and where the hard cases are.

In [ ]:
# For biosecurity grants: extract what would be the org node label under each option
def extract_org_from_grant(grant_title):
    """If org name is blank, try to extract org from grant title (left of ' — ')."""
    if ' — ' in grant_title:
        return grant_title.split(' — ')[0].strip()
    if ' - ' in grant_title:
        return grant_title.split(' - ')[0].strip()
    return grant_title.strip()

bio_scope = bio_scope.copy()
bio_scope['org_resolved'] = bio_scope.apply(
    lambda r: r['Organization Name'].strip()
              if pd.notna(r['Organization Name']) and str(r['Organization Name']).strip() not in ('', 'nan')
              else extract_org_from_grant(r['Grant']),
    axis=1
)

print(f'Unique org_resolved values in biosecurity scope: {bio_scope["org_resolved"].nunique()}')
print(f'\nRows where org was blank and we fell back to grant-title extraction:')
fallback = bio_scope[bio_scope['Organization Name'].isna() | (bio_scope['Organization Name'].str.strip() == '')]
fallback[['org_resolved', 'Grant', 'Amount']].head(15)

## 4. Alias / entity resolution — regex as a unifier, not a filter

Some organisations appear under slightly different names in different rows. Regex should be used to **detect these and merge them to a canonical name** — not to decide whether a row is included at all.

Examples of aliases we need to resolve:

In [ ]:
# Show all unique org names that look like they might be the same entity
all_bio_orgs = sorted(bio_scope['org_resolved'].dropna().unique())

# Manually spotted alias groups from reviewing the data
KNOWN_ALIASES = [
    {
        'canonical': 'RAND Corporation',
        'aliases': ['RAND', 'RAND Corporation'],
        'evidence': 'CSV has both "RAND" and "RAND Corporation" as org names for the same entity'
    },
    {
        'canonical': 'Johns Hopkins Center for Health Security',
        'aliases': ['Johns Hopkins Center for Health Security', 'John Hopkins Center for Health Security'],
        'evidence': 'One row has a typo: "John" instead of "Johns"'
    },
    {
        'canonical': 'SecureBio',
        'aliases': ['SecureBio'],
        'evidence': 'Previously also used "Nucleic Acid Observatory" project name — check grant titles'
    },
    {
        'canonical': 'Georgetown CSET',
        'aliases': ['Georgetown University', 'Center for Security and Emerging Technology'],
        'evidence': 'CSET is at Georgetown; some grants go to the university, some directly to CSET'
    },
    {
        'canonical': 'iGEM Foundation',
        'aliases': ['International Genetically Engineered Machine Foundation',
                    'iGem,International Genetically Engineered Machine Foundation',
                    'iGEM'],
        'evidence': 'Three different spellings across CSV rows'
    },
    {
        'canonical': 'MIT Media Lab (Esvelt group)',
        'aliases': ['Massachusetts Institute of Technology Media Lab',
                    'Massachusetts Institute of Technology',
                    'Berkeley Existential Risk Initiative'],  # BERI-funded Esvelt research at MIT
        'evidence': 'Esvelt grants sometimes routed through BERI, sometimes direct to MIT Media Lab'
    },
]

for group in KNOWN_ALIASES:
    print(f"CANONICAL: {group['canonical']}")
    print(f"  Aliases: {group['aliases']}")
    print(f"  Why: {group['evidence']}")
    # Show actual rows matching any alias
    mask = bio_scope['org_resolved'].isin(group['aliases'])
    print(f"  Rows in biosecurity CSV: {mask.sum()}")
    if mask.sum() > 0:
        for _, r in bio_scope[mask].iterrows():
            print(f"    [{r['org_resolved']:45s}] {r['Amount']:>12s}  {r['Date']:8s}  {r['Grant'][:60]}")
    print()

In [ ]:
# Check: are there any other possible duplicates we haven't caught yet?
# Simple check: org names that share the first 8 characters
from collections import defaultdict
prefix_groups = defaultdict(list)
for org in all_bio_orgs:
    prefix_groups[org[:12].lower()].append(org)

print('Org names sharing a common 12-char prefix (potential aliases):')
for prefix, orgs in sorted(prefix_groups.items()):
    if len(orgs) > 1:
        print(f'  Prefix "{prefix}": {orgs}')

## 5. The 'grant title as node label' question

For rows where the grant recipient is a **person** (not an institution), or where the grant is a pooled fund with no clear org, should we preserve the full grant title?

e.g. `"Jake Pencharz — Metagenomic Sequencing Research"` vs just `"Jake Pencharz"`

**Arguments for preserving the full title:**
- "Jake Pencharz" tells you nothing about why OP funded them — the purpose is in the grant title
- Pooled grants like `"Funding for Projects to Estimate Biological Risk"` have no org name at all
- Two grants to the same person for different projects should arguably be different nodes

**Arguments for org-only:**
- Cleaner graph; easier to see "OP → SecureBio" flow
- Purpose already stored as edge attribute `grant_title`

**Recommendation:** Use org name as the node label, but store the full grant title on the **edge**. When a row has no meaningful org name (blank or looks like a pooled description), use the full grant title as both the node label AND mark it as `"org_type": "grant_pool"` or `"individual"`.

In [ ]:
# Categorise each biosecurity row by entity type
# IMPORTANT: institution check must come FIRST to avoid false person-name matches
# (e.g. "Blueprint Biosecurity", "Gryphon Scientific", "Longview Philanthropy"
#  are two-word capitalised names that would otherwise match the person regex)

person_re2 = re.compile(r'^[A-Z][a-zA-Z\-]+\.? [A-Z][a-zA-Z\-]+\.?$')
pooled_re  = re.compile(
    r'^(Funding for|Early-Career Funding|Open Philanthropy|Scholarship|'
    r'Biosecurity Fund|Travel Grant|Biosecurity Fellow)',
    re.I
)

# Keywords that indicate an org/institution, NOT a person
INST_KW = [
    'University', 'Institute', 'Institution',      # catches Smithsonian Institution
    'College', 'School',
    'Foundation', 'Center', 'Centre', 'Commission', 'Council',
    'Association', 'Society', 'Organisation', 'Organization',
    'Research', 'Sciences', 'Biosciences', 'Scientific',
    'Philanthropy', 'Laboratory', 'Labs', 'Lab',
    'Network', 'Alliance', 'Entrepreneurship', 'Observatory',
    'Endowment', 'Partners', 'Partnership', 'Panel', 'Coalition',
    'Committee', 'Program', 'Initiative', 'Academy',
    'Ltd', 'Inc', 'Corp', 'Consulting', 'Strategy',
    'Policy', 'Studies', 'Affairs', 'Technology', 'Technologies',
    'Bio',       # catches: Blueprint Biosecurity, Synonym Bio, Biosecure…
    'Security',  # catches: Blueprint Biosecurity, SecureBio…
    'Shield',    # catches: Global Shield
    'Bioscience', 'Biodefense', 'Genomic', 'Diagnostics',
]

# Specific known org names that don't contain any of the above keywords
KNOWN_ORG_NAMES = {
    'Global Shield',         # advocacy org
    'Wilton Park',           # UK gov conference centre/forum
    'Longview Philanthropy', # has 'Philanthropy' → already caught, kept for clarity
    'KU Leuven',             # Belgian university (KU = Katholieke Universiteit)
}

def classify_entity(org_resolved):
    s = str(org_resolved).strip() if org_resolved else ''
    if not s:
        return 'unknown'
    # 1. Institution check FIRST (prevents false person matches)
    if any(kw in s for kw in INST_KW):
        return 'institution'
    # 2. Known two-word org names that don't contain an institution keyword
    if s in KNOWN_ORG_NAMES:
        return 'org'
    # 3. Pooled / anonymous grant funds
    if pooled_re.match(s):
        return 'pooled_grant'
    # 4. Only NOW check if it looks like a personal name
    if person_re2.match(s):
        return 'individual'
    # 5. Everything else = org (company / NGO / think-tank)
    return 'org'

bio_scope = bio_scope.copy()
bio_scope['entity_type'] = bio_scope['org_resolved'].apply(classify_entity)

print('Entity type classification for biosecurity-scope rows:')
print(bio_scope['entity_type'].value_counts())
print()
print('Individuals (should be real people, not org names):')
for _, r in bio_scope[bio_scope['entity_type'] == 'individual'].iterrows():
    print(f'  org: "{r["org_resolved"]:<30s}"  grant: "{r["Grant"][:55]}"  {r["Amount"]}')
print()
print('Pooled grants:')
for _, r in bio_scope[bio_scope['entity_type'] == 'pooled_grant'].iterrows():
    print(f'  org_resolved: "{r["org_resolved"][:50]}"  →  node_label: "{r["Grant"][:55]}"')
print()
print('Unknown:')
unknowns = bio_scope[bio_scope['entity_type'] == 'unknown']
if len(unknowns) == 0:
    print('  (none)')
for _, r in unknowns.iterrows():
    print(f'  grant: "{r["Grant"][:65]}"')

## 6. Proposed clean node extraction logic

Here is the logic we'd use for the next iteration. **Review this carefully before running the pipeline.**

Rules:
1. `org_resolved` = `Organization Name` if not blank, else extract the part before ` — ` in `Grant`
2. Apply alias table to normalise known duplicates (RAND/RAND Corporation, JHU typos, iGEM variants)
3. Classify entity type: institution / org / individual / pooled_grant
4. For `individual` and `pooled_grant`: use full grant title as node label (preserves context)
5. For `institution` / `org`: use org name as node label; store grant title on the edge
6. Node ID = stable slug of the canonical label

Below we preview what the clean node list looks like.

In [ ]:
# Alias table: maps any variant → canonical name
# ONLY covers genuine same-entity aliases (typos, abbreviations, project-name vs org-name)
# NOT a filter — all rows stay in the dataset regardless of whether they match
ALIAS_TABLE = {
    # RAND
    'RAND':                                                           'RAND Corporation',
    # JHU typo (one row has "John" instead of "Johns")
    'John Hopkins Center for Health Security':                        'Johns Hopkins Center for Health Security',
    # iGEM three spellings
    'International Genetically Engineered Machine Foundation':        'iGEM Foundation',
    'iGem,International Genetically Engineered Machine Foundation':   'iGEM Foundation',
    'iGEM':                                                           'iGEM Foundation',
    # BERI full name vs acronym
    'Berkeley Existential Risk Initiative':                           'BERI (Berkeley Existential Risk Initiative)',
    # Biosecure / Biosecure Ltd (same org, two names)
    'Biosecure':                                                      'Biosecure Ltd',
}

def apply_alias(name):
    return ALIAS_TABLE.get(name, name)

def node_label(row):
    """
    Final node label for a biosecurity grant row.
    - Individuals and pooled grants: use full grant title (preserves context)
    - All other entity types: use canonical org name (clean graph, detail on edge)
    """
    etype = row['entity_type']
    if etype in ('individual', 'pooled_grant', 'unknown'):
        return row['Grant'].strip()
    return apply_alias(row['org_resolved'])

bio_scope = bio_scope.copy()
bio_scope['node_label']    = bio_scope.apply(node_label, axis=1)
bio_scope['org_canonical'] = bio_scope['org_resolved'].apply(apply_alias)

# Summary of unique nodes
unique_nodes = (
    bio_scope[['node_label', 'entity_type', 'org_canonical']]
    .drop_duplicates('node_label')
    .sort_values(['entity_type', 'node_label'])
    .reset_index(drop=True)
)

print(f'Unique nodes to be created from biosecurity grants: {len(unique_nodes)}')
for etype in ['institution', 'org', 'individual', 'pooled_grant', 'unknown']:
    n = (unique_nodes['entity_type'] == etype).sum()
    print(f'  {etype:15s}: {n}')
print()
print(unique_nodes.to_string())

In [ ]:
# Double-check: any remaining duplicates after alias resolution?
dup_labels = bio_scope.groupby('node_label')['Grant'].count()
dup_labels = dup_labels[dup_labels > 1].sort_values(ascending=False)
print(f'Node labels with multiple grants (expected — these are multi-grant orgs):')
dup_labels.head(20)

In [ ]:
# Alias validation: for institution/org nodes, same canonical name → same node ID?
# (individuals and pooled grants intentionally get distinct node IDs per grant)
def make_slug(s):
    return re.sub(r'[^a-z0-9]+', '_', s.lower()).strip('_')[:60]

bio_scope['node_id_proposed'] = bio_scope['node_label'].apply(make_slug)

# Only check org/institution types — individuals/pooled grants are expected to have multiple IDs
check_rows = bio_scope[bio_scope['entity_type'].isin(['institution', 'org'])]
canon_to_ids = {}
for _, r in check_rows.iterrows():
    canon = r['org_canonical']
    nid   = r['node_id_proposed']
    canon_to_ids.setdefault(canon, set()).add(nid)

issues = {c: ids for c, ids in canon_to_ids.items() if len(ids) > 1}

if issues:
    print('⚠️  Org/institution entities with multiple proposed node IDs (need alias fix):')
    for canon, ids in sorted(issues.items()):
        print(f'\n  Canonical: "{canon}"')
        for nid in sorted(ids):
            matching = bio_scope[(bio_scope['org_canonical'] == canon) & (bio_scope['node_id_proposed'] == nid)]
            for _, r in matching.drop_duplicates('node_id_proposed').iterrows():
                print(f'    id: {nid}')
                print(f'    org_resolved: {r["org_resolved"]}')
else:
    print('✅ No duplicate node IDs for org/institution nodes after alias resolution — entity table looks clean.')

print()
print('(Note: individual and pooled_grant rows intentionally have one node per grant title)')
multi_check = bio_scope[bio_scope['entity_type'].isin(['individual','pooled_grant'])]
print(f'Individual grants: {len(multi_check[multi_check["entity_type"]=="individual"])} rows → '
      f'{multi_check[multi_check["entity_type"]=="individual"]["node_label"].nunique()} unique nodes')
print(f'Pooled grants:     {len(multi_check[multi_check["entity_type"]=="pooled_grant"])} rows → '
      f'{multi_check[multi_check["entity_type"]=="pooled_grant"]["node_label"].nunique()} unique nodes')

<cell_type>markdown</cell_type>## 7. Summary and decisions needed

### ✅ What's clean

| Check | Result |
|---|---|
| NaN org names handled | ✅ `pd.notna()` guard in `org_resolved` |
| Person regex false positives | ✅ Fixed by checking INST_KW **before** person regex |
| KU Leuven (Belgian university) | ✅ In `KNOWN_ORG_NAMES` |
| Smithsonian Institution | ✅ Added `'Institution'` to `INST_KW` |
| Alias validation | ✅ 0 duplicate node IDs for org/institution nodes |
| iGEM 3 spellings → single node | ✅ ALIAS_TABLE |
| RAND / RAND Corporation | ✅ ALIAS_TABLE |
| Biosecure / Biosecure Ltd | ✅ ALIAS_TABLE |

### 📊 Final node count (biosecurity grants only)

- **9 individuals** — real people (Ryan Teo, Tyler Whitmer, Jake Pencharz, Oliver Crook, Cate Hall, Chelsea Liang, Byron Cohen, Andrew Szanton, David Manheim)
- **69 institutions** — universities, think-tanks, NGOs, foundations
- **20 orgs** — companies, startups, smaller organisations
- **2 pooled grants** — OP Biosecurity Scholarships 2022 + 2023
- **Total: 100 unique nodes** from 164 biosecurity rows

---

### 🤔 Decisions still needed before updating the pipeline

**1. Include individuals or not?**
The 9 individual grants (mostly small: $3K–$125K) add personal-name nodes like
`"Ryan Teo — Publishing Fees for BWC Paper"`. Worth including for completeness,
or filter them out to keep the graph focused on organisations?

**2. Classification edge cases** — some "institution"-typed nodes look more like pooled grant pools:
- `"Biosecurity Fellowships"` (blank org, extracted from grant title — $520K)
- `"Biosecurity Funding for Individuals"` (org_resolved extracted — $540K)
- `"Funding for Projects to Estimate Biological Risk"` (blank org — $1.4M)
- `"Open Philanthropy Technology Policy Fellowship (2022)"` (blank org — $2.9M)
- `"Travel Grants for the Biological Weapons Convention Review Conference (2022)"` ($48K)
- `"Early-Career Funding for Global Catastrophic Biological Risks"` (3 rows, ~$4.1M total)

These are real grant pools — should they be their own pooled_grant nodes,
or is institution classification fine?

**3. Alias table complete?**
Run Cell 7 — it should show ✅. If any ⚠️ entries appear, add them to `ALIAS_TABLE`.

---

### ✅ Pipeline update checklist (after you approve the above)

- [ ] Copy `ALIAS_TABLE`, `INST_KW`, `KNOWN_ORG_NAMES`, `classify_entity()`, `node_label()` into `s2_extract_op_grants.py`
- [ ] Store full grant title as `grant_title` on the **edge**, not the node
- [ ] Set `org_subtype` field on each node based on entity_type
- [ ] Decide on individuals (include or exclude?)
- [ ] Re-run pipeline: `s2_extract_op_grants.py` → `s2_validate_op.py` → `s2_merge_op.py`